# Analysis of random corrupted ICL demonstrations

This notebook reproduces the results of section 4.1 of the paper.
Before running this notebook you need:
1. Results of in-context NED for all dataset, random corruption schemes
2. Results of zero-shot and 10-shot NED for all datasets.

For more informatin on on how to run these experiments, please refer to the README file.

In [ ]:
import json
import os
import itertools

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from hydra import compose, initialize
from omegaconf import OmegaConf

In [ ]:
seed = 12345
results_folder = "outputs"
dataset_list = ['chemprotgene', 'chemprotchem', 'bc5chem', 'bc5disease', 'bc2gm']
random_corruption_schemes = ["random-id-labels", "swapped-id-labels", "random-ood-labels", "random-ood-labels-from-text", "corrupted-ood-text", "corrupted-and-shuffled-ood-text", "corrupted-ood-text-and-labels", "corrupted-and-shuffled-ood-text-and-labels", None]
demo_retrieval='knn'

In [ ]:
setups = []
for dataset in dataset_list:
    for corruption_type in random_corruption_schemes:
        setups.append({
            'dataset': dataset,
            'corruption': corruption_type,
            'num_shots': 10,
            'experiment_name': corruption_type.replace("-", ' ').camelcase().replace("id", "ID").replace("od", "OOD"),
            'results_path': ''
        })

    setups.extend([
        {
            'dataset': dataset,
            'corruption': None,
            'num_shots': 10,
            'experiment_name': 'Gold Label',
            'results_path': ''
        },
        {
            'dataset': dataset,
            'corruption': None,
            'num_shots': 0,
            'experiment_name': 'No Demo',
            'results_path': ''
        }
    ])

sorted_experiments_desc = sorted(os.listdir(results_folder), reverse=True)
for day in sorted_experiments_desc:
    for time in sorted(os.listdir(os.path.join(results_folder, day)), reverse=True):
        with initialize(config_path=os.path.join(results_folder, day, time, '.hydra')):
            cfg = compose(config_name="config")
        
        cfg_dict = OmegaConf.to_container(cfg, resolve=True)
        dataset = cfg_dict['data']['dataset']
        corruption = cfg_dict.get('corruption_type', {})
        if corruption is not None:
            corruption = corruption.get('name', None)
        num_shots = cfg_dict['demonstration_retrieval']['num_shots']

        if dataset in dataset_list and corruption in random_corruption_schemes and cfg_dict['demonstration_retrieval'].get('method', None) == demo_retrieval:
            for setup in setups:
                if setup['dataset'] == dataset and setup['corruption'] == corruption and setup['num_shots'] == num_shots and setup['results_path'] == '':
                    setup['results_path'] = os.path.join(results_folder, day, time, 'ner_iob_evaluation.json')
                    break

for setup in setups:
    if setup['results_path'] == '':
        print(f"Missing results for {setup['dataset']} {setup['corruption']}")
        continue

In [ ]:
results_dict = {
    "dataset": [],
    "experiment_name": [],
    "Precision": [],
    "Recall": [],
    "Micro F1": [],
}
for setup in setups:
    results = json.load(open(setup['results_path'], "r"))
    results_dict["dataset"].append(setup['dataset'])
    results_dict["experiment_name"].append(setup['experiment_name'])
    results_dict["prc"].append(results["micro avg"]['precision'])
    results_dict["rec"].append(results["micro avg"]['recall'])
    results_dict["f1"].append(results["micro avg"]['f1-score'])
    
results_df = pd.DataFrame(results_dict)

In [ ]:
def plot_random_corrupt_analysis(results_df, model, metric):
    
    fontsize=70
    plt.rcParams['hatch.linewidth'] = 1
    _, ax = plt.subplots(figsize=(30, 7))

    palette = [
        "#A7C7E7",  # Pastel Blue
        "#F8E5A5",  # Pastel Yellow
        "#B5EAD7",  # Pastel Green
        "#A0D8B3",  #  Muted Pastel Green
        "#F6C7A3",  # Pastel Orange
        "#E8B494",  # Muted Pastel Orange
        "#D6B3E7",  # Pastel Purple
        "#C9A2D8",  # Muted Pastel Purple 
        "#F2B2C2",  # Pastel Pink
        "#E3A3B5"   # Muted Pastel Pink
    ]
    palette = ['lightgray', 'gray', 'darkcyan', '#40B9B9', 'cornflowerblue', '#A2B9F2', 'orangered', '#FF8566', 'darkorange', '#FFBE66']

    ax = sns.barplot(x="dataset", y=metric, hue="demo data", data=results_df, ax=ax, edgecolor=".2", linewidth=2.5, palette= palette, alpha=1)

    hatches = itertools.cycle(['', 'o', 'x', '*', '.', '\\', '+', '/', '-', '/'])
    hatch = '/'
    num_locations = len(results_df['experiment_name'].unique())
    location_w_hatch = []
    for i in [1,3,5,7,9]:
        location_w_hatch.extend([num_locations*i, num_locations*i+1, num_locations*i+2, num_locations*i+3, num_locations*i+4])
    
    for i, patch in enumerate(ax.patches):
        if i % (num_locations) == 0:
            hatch = next(hatches)
        patch.set_hatch(hatch)
    
    
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0, horizontalalignment='center', fontsize=fontsize-5)
    ax.set_xlabel("")

    ax.set_ylabel(metric, fontsize=fontsize-5)
    if model == 'gpt-3.5-turbo':
        ax.set_ylim(0.0, 0.9)
    elif model == 'mistral':
        ax.set_ylim(0.0, 0.62)

    ax.tick_params(axis='y', labelsize=fontsize-5)
    
    for y in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
        ax.axhline(y = y, color = 'gray', linestyle = '--')
    
    plt.legend(bbox_to_anchor=(-0.02, 1.3), loc=2, borderaxespad=0., fontsize=fontsize-10, ncol=5)
    plt.tight_layout()
    plt.xlabel("")
    plt.show()

In [ ]:
plot_random_corrupt_analysis(results_df, 'mistral', 'Micro F1')